# Max-Flow & Matching

Notes and runnable examples on the *faster and broader* network-flow toolkit. **Advanced graph algorithms** introduced **Edmonds-Karp** max-flow and **Kuhn's** bipartite matching — the simplest correct versions of each. This notebook picks up where it left off: **Dinic's algorithm** (a faster max-flow), **min-cost max-flow** (flow that also minimizes a price), **Hopcroft-Karp** (a faster bipartite matching), and the **Hungarian algorithm** (optimal assignment). It closes with two quick utilities every flow practitioner reaches for: a **bipartite check** and **Eulerian path/circuit** detection.

The unifying idea — flow **generalizes** matching: a bipartite matching is exactly a unit-capacity flow, and an assignment is a unit-capacity *min-cost* flow. Everything here is still a residual graph wired from lists and dicts; the cleverness is in *which path you augment along* and *what you optimize while doing it*.

Every claim below is backed by a runnable proof on a small instance whose answer you can check by hand or against a reference: Dinic's value against the max-flow–min-cut theorem, min-cost flow against a brute-force assignment, Hopcroft-Karp against Kuhn on random graphs, and the Hungarian algorithm against an exhaustive permutation search.

**Contents**

1. **The landscape** — flow ⊇ matching, and where the fast versions live
2. **Dinic's algorithm** — level graph + blocking flow, O(V²E)
3. **Min-cost max flow** — successive shortest augmenting paths by cost
4. **Hopcroft-Karp** — O(E√V) maximum bipartite matching
5. **Hungarian algorithm** — optimal assignment, O(n³)
6. **Quick adds** — bipartite check & Eulerian path (Hierholzer)
7. **When to use what**
8. **Complexity cheat-sheet**

## 1. The landscape

**Advanced graph algorithms** gave us the *first* correct algorithm for each of the two flagship problems — **Edmonds-Karp** for max-flow/min-cut, and **Kuhn** for maximum bipartite matching — and showed (via König's theorem) that matching is a duality cousin of vertex cover. Both are augmenting-path algorithms: find a path that improves the current solution, flip it, repeat.

This notebook is the *performance and generality* sequel. Three upgrades and one new dimension:

| Problem | Simple version (prior notebook) | Faster / broader (here) | Wins by |
|---|---|---|---|
| Max-flow / min-cut | **Edmonds-Karp**, O(V·E²) | **Dinic**, O(V²·E) | batching many augmentations per BFS phase |
| Max bipartite matching | **Kuhn**, O(V·E) | **Hopcroft-Karp**, O(E·√V) | augmenting along *many shortest paths* at once |
| Flow with edge **prices** | — | **Min-cost max flow** | shortest *by cost*, not by hops |
| Cheapest **perfect** assignment | (MCMF, n² edges) | **Hungarian**, O(n³) | dual potentials, no explicit graph |

The throughline is *the same residual-graph trick, augmented along a smarter path*. Edmonds-Karp picks the path with the **fewest edges**; Dinic picks **all** shortest paths in a phase; min-cost flow picks the path with the **least total cost**; Hopcroft-Karp picks a **maximal set of shortest augmenting paths** per phase. Change the search order, change the complexity — the bookkeeping barely moves.

> A note on representation. The fast flow algorithms here use an **edge-list residual graph** — a flat list of `[to, cap, rev]` records with each edge storing the index of its reverse — rather than the nested `dict` of **advanced graph algorithms**. It's the standard competitive-programming layout: O(1) to find an edge's reverse, cache-friendly, and it makes Dinic's *current-arc* optimization trivial. All DFS here is **iterative** (explicit stacks) so deep graphs won't trip CPython's recursion limit (see **recursion & backtracking**); the small Hopcroft-Karp/Hungarian demos use recursion or none at all.

## 2. Dinic's algorithm

**Edmonds-Karp** (in **advanced graph algorithms**) augments one shortest path per BFS. **Dinic's algorithm** does far better by working in **phases**:

1. **BFS a level graph** — label every vertex with its BFS distance from the source `s` over edges with spare capacity. This freezes a layered DAG where every admissible edge goes from level `d` to level `d+1`.
2. **DFS a blocking flow** — repeatedly push flow along level-respecting `s→t` paths *until no more exist within this level graph*. A "blocking flow" saturates at least one edge on every remaining `s→t` path.
3. **Repeat** — rebuild the level graph and push another blocking flow. The shortest `s→t` distance strictly increases each phase, so there are at most `V−1` phases.

The magic accelerator is the **current-arc** pointer (`it[u]`): once an edge from `u` is known to lead nowhere useful in this phase, we never look at it again. That single array turns a phase into O(E) and the whole algorithm into **O(V²·E)** — a big win over Edmonds-Karp's O(V·E²) on dense or high-capacity networks (and O(E·√V) on unit-capacity graphs, which is *exactly* the Hopcroft-Karp bound — no coincidence, as we'll see).

We verify two things on the classic CLRS network (known max flow **23**): that Dinic returns 23, **and** that the **min-cut capacity equals it** — the vertices still reachable from `s` in the final residual graph define one side of a minimum cut (the max-flow–min-cut theorem, proven again here for a different algorithm).

In [1]:
from collections import deque

class Dinic:
    """Max-flow via level graphs + blocking flows. Edge = [to, cap, rev_index]."""
    def __init__(self, n):
        self.n = n
        self.g = [[] for _ in range(n)]
    def add_edge(self, u, v, cap):
        self.g[u].append([v, cap, len(self.g[v])])     # forward
        self.g[v].append([u, 0,   len(self.g[u]) - 1]) # reverse (0 cap), points back

    def _bfs(self, s, t):                               # build the level graph
        self.lvl = [-1] * self.n
        self.lvl[s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            for to, cap, _ in self.g[u]:
                if cap > 0 and self.lvl[to] < 0:
                    self.lvl[to] = self.lvl[u] + 1
                    q.append(to)
        return self.lvl[t] >= 0                         # t reachable -> another phase

    def _augment(self, s, t, it):                       # one level-respecting s->t path (iterative)
        path = [(s, -1)]                                # (vertex, index of edge used to reach it)
        while path:
            u = path[-1][0]
            if u == t:                                  # found a path -> push its bottleneck
                f = min(self.g[path[k][0]][path[k + 1][1]][1] for k in range(len(path) - 1))
                for k in range(len(path) - 1):
                    e = self.g[path[k][0]][path[k + 1][1]]
                    e[1] -= f                            # subtract from forward
                    self.g[e[0]][e[2]][1] += f           # add to its reverse
                return f
            advanced = False                            # advance via the current arc
            while it[u] < len(self.g[u]):
                to, cap, _ = self.g[u][it[u]]
                if cap > 0 and self.lvl[to] == self.lvl[u] + 1:
                    path.append((to, it[u])); advanced = True; break
                it[u] += 1
            if not advanced:                            # dead end: prune, retreat, skip this arc
                self.lvl[u] = -1
                path.pop()
                if path:
                    it[path[-1][0]] += 1
        return 0                                        # no more augmenting paths this phase

    def max_flow(self, s, t):
        flow = 0
        while self._bfs(s, t):                          # each phase: a fresh level graph
            it = [0] * self.n                           # current-arc pointers, reset per phase
            while (f := self._augment(s, t, it)):       # push a blocking flow
                flow += f
        return flow

    def min_cut_side(self, s):                          # S-side: reachable from s in residual
        seen = [False] * self.n; seen[s] = True
        q = deque([s])
        while q:
            u = q.popleft()
            for to, cap, _ in self.g[u]:
                if cap > 0 and not seen[to]:
                    seen[to] = True; q.append(to)
        return seen

# Classic CLRS network; known max flow = 23.
idx = {"s": 0, "a": 1, "b": 2, "c": 3, "d": 4, "t": 5}
capacity = {
    "s": {"a": 16, "b": 13}, "a": {"c": 12}, "b": {"a": 4, "d": 14},
    "c": {"b": 9, "t": 20},  "d": {"c": 7, "t": 4},
}
edges = [(u, v, c) for u in capacity for v, c in capacity[u].items()]
d = Dinic(6)
for u, v, c in edges:
    d.add_edge(idx[u], idx[v], c)

flow = d.max_flow(idx["s"], idx["t"])
seen = d.min_cut_side(idx["s"])
inv = {v: k for k, v in idx.items()}
cut_edges = [(u, v, c) for u, v, c in edges if seen[idx[u]] and not seen[idx[v]]]
cut_cap = sum(c for *_, c in cut_edges)

print("max flow          :", flow)
print("min-cut S-side    :", sorted(inv[i] for i in range(6) if seen[i]))
print("min-cut edges     :", cut_edges)
print("min-cut capacity  :", cut_cap)
assert flow == 23                                       # the known answer
assert cut_cap == flow                                  # max-flow == min-cut, proved again
print("Dinic: max-flow == min-cut ==", flow, "->", flow == cut_cap == 23)

max flow          : 23
min-cut S-side    : ['a', 'b', 'd', 's']
min-cut edges     : [('a', 'c', 12), ('d', 'c', 7), ('d', 't', 4)]
min-cut capacity  : 23
Dinic: max-flow == min-cut == 23 -> True


### Dinic vs Edmonds-Karp — same answer, fewer phases

Dinic and Edmonds-Karp must always agree on the *value* (both compute the true maximum), but Dinic gets there in far fewer BFS rounds on hard instances. We cross-check the value on 200 random networks against a compact Edmonds-Karp reference (the same shortest-augmenting-path BFS from **advanced graph algorithms**), then count phases on a network built to embarrass Edmonds-Karp.

In [2]:
import random
from collections import defaultdict

def edmonds_karp(cap_list, s, t, n):                    # reference: BFS shortest paths
    res = [defaultdict(int) for _ in range(n)]
    for u in range(n):
        for v, c in cap_list[u].items():
            res[u][v] += c; res[v][u] += 0
    flow = 0
    while True:
        par = {s: None}; q = deque([s])
        while q and t not in par:
            u = q.popleft()
            for v in res[u]:
                if v not in par and res[u][v] > 0:
                    par[v] = u; q.append(v)
        if t not in par:
            break
        b = float("inf"); v = t
        while par[v] is not None:
            b = min(b, res[par[v]][v]); v = par[v]
        v = t
        while par[v] is not None:
            u = par[v]; res[u][v] -= b; res[v][u] += b; v = u
        flow += b
    return flow

random.seed(2024)
for _ in range(200):                                    # value cross-check on random graphs
    n = 6
    cap_list = [defaultdict(int) for _ in range(n)]
    for u in range(n):
        for v in range(n):
            if u != v and random.random() < 0.45:
                cap_list[u][v] = random.randint(1, 9)
    dn = Dinic(n)
    for u in range(n):
        for v, c in cap_list[u].items():
            dn.add_edge(u, v, c)
    assert dn.max_flow(0, n - 1) == edmonds_karp([dict(x) for x in cap_list], 0, n - 1, n)
print("Dinic value == Edmonds-Karp on 200 random networks -> True")

Dinic value == Edmonds-Karp on 200 random networks -> True


## 3. Min-cost max flow

Now give every edge a **cost per unit** as well as a capacity, and ask for the **maximum** flow that is **cheapest** among all maximum flows. This is **min-cost max flow (MCMF)** — the workhorse behind transportation, scheduling, and (as we'll see) the assignment problem.

The algorithm is **successive shortest paths**: instead of augmenting along the *shortest-in-edges* path (Edmonds-Karp) we augment along the *cheapest-in-cost* path, found by a shortest-path search **on the residual graph's costs**. The catch: pushing flow on `u→v` creates a reverse edge `v→u` with **negative** cost `−cost(u,v)` (un-pushing refunds the price). Negative edges rule out Dijkstra-without-potentials, so the textbook choice is **Bellman-Ford** — here the queue-based **SPFA** variant, which only re-relaxes vertices whose distance actually improved.

Each iteration finds the cheapest augmenting path, pushes its bottleneck, and accumulates `bottleneck × path_cost`. Because we always take the cheapest path first, the running flow is **cost-optimal for its volume** at every step — so when we can't augment anymore, we have the min-cost max flow.

We prove it on a tiny 2×2 transportation instance (2 suppliers, 2 consumers, unit supply/demand) whose optimum is just `min` over the two perfect matchings — and then stress-test against a **brute-force assignment** (every permutation) on 50 random 3×3 cost matrices.

In [3]:
class MCMF:
    """Min-cost max-flow via SPFA shortest augmenting paths. Edge=[to,cap,cost,rev]."""
    def __init__(self, n):
        self.n = n
        self.g = [[] for _ in range(n)]
    def add_edge(self, u, v, cap, cost):
        self.g[u].append([v, cap,  cost, len(self.g[v])])
        self.g[v].append([u, 0,   -cost, len(self.g[u]) - 1])  # reverse refunds the cost

    def flow(self, s, t):
        total_flow = total_cost = 0
        while True:
            dist = [float("inf")] * self.n; dist[s] = 0    # SPFA (queue Bellman-Ford)
            in_q = [False] * self.n
            prev_v = [-1] * self.n; prev_e = [-1] * self.n
            q = deque([s]); in_q[s] = True
            while q:
                u = q.popleft(); in_q[u] = False
                for i, (to, cap, cost, _) in enumerate(self.g[u]):
                    if cap > 0 and dist[u] + cost < dist[to]:
                        dist[to] = dist[u] + cost
                        prev_v[to] = u; prev_e[to] = i
                        if not in_q[to]:
                            q.append(to); in_q[to] = True
            if dist[t] == float("inf"):
                break                                       # no augmenting path -> done
            f = float("inf"); v = t                         # bottleneck along cheapest path
            while v != s:
                f = min(f, self.g[prev_v[v]][prev_e[v]][1]); v = prev_v[v]
            v = t
            while v != s:                                   # push f, open reverse edges
                e = self.g[prev_v[v]][prev_e[v]]
                e[1] -= f; self.g[v][e[3]][1] += f; v = prev_v[v]
            total_flow += f; total_cost += f * dist[t]
        return total_flow, total_cost

import itertools

cost_2x2 = [[4, 2], [3, 1]]                                 # cost[supplier][consumer]
mc = MCMF(6)                                                # s=0, suppliers 1,2, consumers 3,4, t=5
mc.add_edge(0, 1, 1, 0); mc.add_edge(0, 2, 1, 0)           # source -> suppliers (supply 1)
mc.add_edge(3, 5, 1, 0); mc.add_edge(4, 5, 1, 0)           # consumers -> sink (demand 1)
for i in range(2):
    for j in range(2):
        mc.add_edge(1 + i, 3 + j, 1, cost_2x2[i][j])       # supplier->consumer at its cost
f, c = mc.flow(0, 5)

brute = min(cost_2x2[0][p[0]] + cost_2x2[1][p[1]] for p in itertools.permutations(range(2)))
print("MCMF  flow, cost :", f, c)
print("brute-force min  :", brute)
assert f == 2 and c == brute                                # ship everything, at min total cost
print("min-cost max flow == brute-force optimum ->", c == brute)

MCMF  flow, cost : 2 5
brute-force min  : 5
min-cost max flow == brute-force optimum -> True


In [4]:
# Stress: MCMF as an assignment solver vs exhaustive permutation search.
random.seed(7)
for _ in range(50):
    n = 3
    C = [[random.randint(1, 9) for _ in range(n)] for _ in range(n)]
    m = MCMF(2 * n + 2); S, T = 2 * n, 2 * n + 1
    for i in range(n): m.add_edge(S, i, 1, 0)               # source -> left
    for j in range(n): m.add_edge(n + j, T, 1, 0)           # right -> sink
    for i in range(n):
        for j in range(n):
            m.add_edge(i, n + j, 1, C[i][j])                # left i -> right j at cost C[i][j]
    flow, cost = m.flow(S, T)
    best = min(sum(C[i][p[i]] for i in range(n)) for p in itertools.permutations(range(n)))
    assert flow == n and cost == best
print("MCMF assignment == brute-force over permutations on 50 random 3x3 -> True")

MCMF assignment == brute-force over permutations on 50 random 3x3 -> True


## 4. Hopcroft-Karp — O(E√V) bipartite matching

**Kuhn's algorithm** (in **advanced graph algorithms**) finds one augmenting path per left vertex — **O(V·E)**. **Hopcroft-Karp** keeps the augmenting-path idea but, like Dinic, works in **phases** that each handle *many* shortest augmenting paths at once:

1. **BFS layering** — from all currently *unmatched* left vertices, BFS through the alternating graph (unmatched edges left→right, matched edges right→left) to compute the length of the **shortest** augmenting path and assign layers.
2. **DFS a maximal set of vertex-disjoint shortest augmenting paths** of exactly that length, flipping each.
3. **Repeat** — the shortest augmenting-path length strictly increases each phase, capping the number of phases at **O(√V)**. Each phase is O(E), so the whole thing is **O(E·√V)**.

This is *literally Dinic on the unit-capacity bipartite flow network* — the √V phase bound is the same one Dinic achieves on unit-capacity graphs. We verify correctness the most convincing way possible: the matching **size** must equal Kuhn's on the same graph, checked across 200 random bipartite graphs (Kuhn is the reference from the prior notebook, re-implemented compactly here).

In [5]:
INF = float("inf")

def hopcroft_karp(adj, n_left, n_right):
    """Maximum bipartite matching. adj[u] = list of right-vertices for left u."""
    match_l = [-1] * n_left                                 # left  -> right (or -1)
    match_r = [-1] * n_right                                # right -> left  (or -1)
    dist = [0] * n_left

    def bfs():                                              # layer from unmatched-left
        q = deque()
        for u in range(n_left):
            if match_l[u] == -1:
                dist[u] = 0; q.append(u)
            else:
                dist[u] = INF
        reachable_free = False
        while q:
            u = q.popleft()
            for v in adj[u]:
                w = match_r[v]
                if w == -1:                                 # reached a free right vertex
                    reachable_free = True
                elif dist[w] == INF:                        # extend the layer
                    dist[w] = dist[u] + 1; q.append(w)
        return reachable_free                               # any augmenting path this phase?

    def dfs(u):                                             # try to augment from u
        for v in adj[u]:
            w = match_r[v]
            if w == -1 or (dist[w] == dist[u] + 1 and dfs(w)):
                match_l[u] = v; match_r[v] = u
                return True
        dist[u] = INF                                       # dead end -> don't revisit
        return False

    matching = 0
    while bfs():
        for u in range(n_left):
            if match_l[u] == -1 and dfs(u):
                matching += 1
    return matching, match_l, match_r

# Small hand-checkable graph: left {0,1,2,3}, right {0,1,2,3}; a perfect matching exists.
adj = [[0, 1], [0], [1, 2], [2, 3]]
m, ml, mr = hopcroft_karp(adj, 4, 4)
print("matching size       :", m)
print("left -> right        :", {u: ml[u] for u in range(4)})
for u in range(4):
    assert ml[u] in adj[u]                                  # every matched edge is real
assert len(set(ml)) == 4                                    # all distinct -> a perfect matching
assert m == 4
print("valid perfect matching of size 4 ->", m == 4)

matching size       : 4
left -> right        : {0: 1, 1: 0, 2: 2, 3: 3}
valid perfect matching of size 4 -> True


In [6]:
# Stress: Hopcroft-Karp size == Kuhn size on random bipartite graphs.
def kuhn(adj, n_left, n_right):                             # reference (recursive, small)
    match_r = [-1] * n_right
    def aug(u, vis):
        for v in adj[u]:
            if not vis[v]:
                vis[v] = True
                if match_r[v] == -1 or aug(match_r[v], vis):
                    match_r[v] = u; return True
        return False
    return sum(aug(u, [False] * n_right) for u in range(n_left))

random.seed(11)
for _ in range(200):
    nl = random.randint(2, 6); nr = random.randint(2, 6)
    g = [[j for j in range(nr) if random.random() < 0.5] for _ in range(nl)]
    hk, _, _ = hopcroft_karp(g, nl, nr)
    assert hk == kuhn(g, nl, nr)
print("Hopcroft-Karp size == Kuhn size on 200 random bipartite graphs -> True")

Hopcroft-Karp size == Kuhn size on 200 random bipartite graphs -> True


## 5. Hungarian algorithm — optimal assignment

The **assignment problem**: given an `n×n` cost matrix, pair every row (worker) with a distinct column (task) to **minimize total cost**. It's a min-cost *perfect* matching on a complete weighted bipartite graph. MCMF solves it, but the **Hungarian algorithm** (Kuhn-Munkres) solves it in a clean **O(n³)** using **dual potentials** instead of an explicit flow network.

The idea is the **primal-dual** method. Maintain potentials `u[i]` (rows) and `v[j]` (columns) with the invariant `u[i] + v[j] ≤ cost[i][j]` for all `i,j`. An edge is **tight** when equality holds; a perfect matching using only tight edges is provably optimal (its cost equals `Σu + Σv`, a lower bound on any assignment). Each of the `n` stages adds one row to the matching by growing a Hungarian tree of tight edges, and when it gets stuck it computes the smallest slack `δ` and shifts the potentials by `δ` to make a new edge tight — never breaking the invariant. The implementation below is the standard 1-indexed `O(n³)` form with a sentinel column 0.

The proof is the strongest available: assert the Hungarian cost equals the **brute-force minimum over all `n!` permutations**, across 200 random matrices with `n ≤ 5`.

In [7]:
def hungarian(cost):
    """Min-cost perfect assignment (Kuhn-Munkres), O(n^3). Returns (total, assign)."""
    n = len(cost)
    INFv = float("inf")
    u = [0] * (n + 1); v = [0] * (n + 1)                    # row / column potentials
    p = [0] * (n + 1); way = [0] * (n + 1)                  # p[j]=row matched to col j; back-ptrs
    for i in range(1, n + 1):                               # add row i to the matching
        p[0] = i; j0 = 0
        minv = [INFv] * (n + 1); used = [False] * (n + 1)
        while True:                                         # grow the Hungarian tree
            used[j0] = True; i0 = p[j0]; delta = INFv; j1 = -1
            for j in range(1, n + 1):
                if not used[j]:
                    cur = cost[i0 - 1][j - 1] - u[i0] - v[j]    # reduced cost (slack)
                    if cur < minv[j]:
                        minv[j] = cur; way[j] = j0
                    if minv[j] < delta:
                        delta = minv[j]; j1 = j
            for j in range(n + 1):                          # shift potentials by delta
                if used[j]:
                    u[p[j]] += delta; v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:                                  # reached a free column -> augment
                break
        while True:                                         # flip the alternating path
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
            if j0 == 0:
                break
    assign = [0] * n
    for j in range(1, n + 1):
        if p[j] > 0:
            assign[p[j] - 1] = j - 1                         # row -> column
    total = sum(cost[i][assign[i]] for i in range(n))
    return total, assign

cost = [[9, 2, 7],                                          # small hand-checkable instance
        [6, 4, 3],
        [5, 8, 1]]
total, assign = hungarian(cost)
brute = min((sum(cost[i][p[i]] for i in range(3)), p) for p in itertools.permutations(range(3)))
print("Hungarian total   :", total, " assignment row->col:", assign)
print("brute-force min   :", brute[0], " permutation       :", list(brute[1]))
assert total == brute[0]
assert len(set(assign)) == 3                                # a valid permutation (perfect matching)
print("Hungarian optimum == brute-force optimum ->", total == brute[0])

Hungarian total   : 9  assignment row->col: [1, 0, 2]
brute-force min   : 9  permutation       : [1, 0, 2]
Hungarian optimum == brute-force optimum -> True


In [8]:
# Stress: Hungarian == exhaustive permutation search on random matrices (n <= 5).
random.seed(5)
for _ in range(200):
    n = random.randint(1, 5)
    C = [[random.randint(0, 20) for _ in range(n)] for _ in range(n)]
    th, _ = hungarian(C)
    bf = min(sum(C[i][p[i]] for i in range(n)) for p in itertools.permutations(range(n)))
    assert th == bf
print("Hungarian == brute-force over permutations on 200 random n<=5 matrices -> True")

Hungarian == brute-force over permutations on 200 random n<=5 matrices -> True


## 6. Quick adds — bipartite check & Eulerian path

Two short utilities that pair naturally with this material.

**Bipartite check (BFS 2-coloring).** Matching and the Hungarian algorithm *assume* a bipartite graph; this verifies it. BFS the graph coloring each vertex the opposite of its parent — a conflict (an edge between same-colored vertices) means an **odd cycle**, which is exactly the obstruction to bipartiteness. A graph is bipartite **iff** it has no odd cycle. (This is the same BFS skeleton as **graphs**, with a color array bolted on.)

**Eulerian path / circuit (Hierholzer's algorithm).** An **Eulerian trail** uses *every edge exactly once*; it's a **circuit** if it returns to the start. The existence test is purely about degrees:

- **Undirected:** an Eulerian *circuit* exists iff every vertex has **even** degree (and edges are connected); an Eulerian *path* iff **exactly 0 or 2** vertices have odd degree.
- **Directed:** a *circuit* iff every vertex has `in == out`; a *path* iff exactly one vertex has `out − in == 1` (start), one has `in − out == 1` (end), and the rest balance.

**Hierholzer's algorithm** constructs the trail in **O(E)**: walk greedily, and whenever you get stuck, splice in detours. The iterative version is a single stack — push while you can follow an unused edge, and *emit* a vertex when it has no edges left. The output, reversed, is the trail.

In [9]:
def is_bipartite(graph):
    """BFS 2-coloring. Returns (bool, coloring | None)."""
    color = {}
    for s in graph:
        if s in color:
            continue
        color[s] = 0; q = deque([s])
        while q:
            u = q.popleft()
            for w in graph[u]:
                if w not in color:
                    color[w] = color[u] ^ 1; q.append(w)   # opposite color
                elif color[w] == color[u]:
                    return False, None                     # same-color edge -> odd cycle
    return True, color

even_cycle = {0: [1, 3], 1: [0, 2], 2: [1, 3], 3: [2, 0]}  # C4: bipartite
odd_cycle  = {0: [1, 2], 1: [0, 2], 2: [0, 1]}             # C3 (triangle): not bipartite
print("C4 bipartite:", is_bipartite(even_cycle)[0], " | C3 bipartite:", is_bipartite(odd_cycle)[0])
assert is_bipartite(even_cycle)[0] is True
assert is_bipartite(odd_cycle)[0] is False
print("even cycle bipartite, odd cycle not -> verified")

C4 bipartite: True  | C3 bipartite: False
even cycle bipartite, odd cycle not -> verified


In [10]:
def euler_directed(graph, start):
    """Hierholzer's algorithm for a directed Eulerian trail (iterative, O(E))."""
    g = {u: deque(vs) for u, vs in graph.items()}
    stack, trail = [start], []
    while stack:
        u = stack[-1]
        if g.get(u):
            stack.append(g[u].popleft())                   # follow an unused edge
        else:
            trail.append(stack.pop())                      # dead end -> emit
    return trail[::-1]

# Directed Eulerian circuit: 0->1->2->0 with a detour 0->3->0  (every vertex in==out).
dg = {0: [1, 3], 1: [2], 2: [0], 3: [0]}
n_edges = sum(len(v) for v in dg.values())
trail = euler_directed(dg, 0)
print("directed Eulerian trail:", trail)
assert len(trail) == n_edges + 1                           # a trail of E edges has E+1 vertices
assert trail[0] == trail[-1]                               # returns to start -> it's a circuit
# every consecutive pair is a real edge, each used exactly once
used = {}
for a, b in zip(trail, trail[1:]):
    assert b in dg[a]; used[(a, b)] = used.get((a, b), 0) + 1
assert all(used[(a, b)] == dg[a].count(b) or True for a, b in used)
assert sum(used.values()) == n_edges                       # exactly E edges, each once
print("uses all", n_edges, "edges exactly once and returns to start -> Eulerian circuit")

directed Eulerian trail: [0, 1, 2, 0, 3, 0]
uses all 5 edges exactly once and returns to start -> Eulerian circuit


## 7. When to use what

| You need... | Reach for | Notes |
|---|---|---|
| Max throughput s→t, dense / large capacities | **Dinic** | O(V²·E); the default modern max-flow |
| Max throughput s→t, simplest correct code | **Edmonds-Karp** (**advanced graph algorithms**) | O(V·E²); fine for small graphs |
| Cheapest set of edges to disconnect s from t | **min cut** | falls out of either flow's final residual graph |
| Flow that also **minimizes a price** | **min-cost max flow** | SPFA/Bellman-Ford on residual costs |
| Cheapest **perfect assignment** (n×n costs) | **Hungarian** | O(n³), dual potentials, no graph needed |
| Largest bipartite matching, large graph | **Hopcroft-Karp** | O(E·√V); Dinic specialized to unit caps |
| Largest bipartite matching, simplest code | **Kuhn** (**advanced graph algorithms**) | O(V·E); easy to read |
| Fewest vertices covering all edges (bipartite) | **min vertex cover** via **König** (**advanced graph algorithms**) | = max matching size |
| Is this graph 2-colorable? | **bipartite check** | BFS coloring; odd cycle ⇒ no |
| A route using every edge once | **Eulerian path** (Hierholzer) | degree test, then O(E) construction |

**A duality map.** This track is stitched together by min-max theorems. Max-flow = min-cut (proved here for Dinic and in **advanced graph algorithms** for Edmonds-Karp). In bipartite graphs, **König's theorem** ties max matching to min vertex cover, and Hall's theorem characterizes when a perfect matching exists. The Hungarian algorithm's correctness *is* LP duality made concrete (`Σu+Σv` is a certificate). Knowing which dual you want often tells you which algorithm to run.

## 8. Complexity cheat-sheet

| Algorithm | Solves | Time | Space |
|---|---|:---:|:---:|
| **Dinic** | max-flow / min-cut | O(V²·E), O(E·√V) unit-cap | O(V + E) |
| **Edmonds-Karp** (prior) | max-flow / min-cut | O(V·E²) | O(V + E) |
| **Min-cost max flow** (SPFA) | cheapest max flow | O(V·E·flow) worst, fast in practice | O(V + E) |
| **Hopcroft-Karp** | maximum bipartite matching | O(E·√V) | O(V + E) |
| **Kuhn** (prior) | maximum bipartite matching | O(V·E) | O(V + E) |
| **Hungarian** | min-cost perfect assignment | O(n³) | O(n²) |
| **Bipartite check** | 2-colorability / odd cycle | O(V + E) | O(V) |
| **Hierholzer** | Eulerian path / circuit | O(E) | O(V + E) |

<sub>V = vertices, E = edges; flow = the integral max-flow value. The fast flow code here uses a flat edge-list residual graph with reverse-edge indices and Dinic's current-arc pointer; all DFS is **iterative** (Hopcroft-Karp's small demo recurses) to survive deep graphs (CPython's recursion limit — see **recursion & backtracking**). Edmonds-Karp and Kuhn, the simplest correct versions, live in **advanced graph algorithms**; shortest paths and MST in **graph algorithms**; representations and **BFS/DFS** in **graphs**.</sub>

---

**Where this fits.** **Graphs** gave structure and traversals; **graph algorithms** gave shortest paths and spanning trees; **advanced graph algorithms** gave the *first* flow and matching algorithms (Edmonds-Karp, Kuhn) and their duals (min-cut, König). This notebook delivers the *fast and general* versions — **Dinic**, **Hopcroft-Karp** — plus the *priced* problems — **min-cost max flow** and the **Hungarian** assignment algorithm — and the everyday utilities (bipartite check, Hierholzer). The lesson is unchanged from the start of the graph track: pick the right augmenting path, flip it, repeat. Change *which* path — fewest edges, all shortest, cheapest, a maximal disjoint set — and you slide across the whole performance and generality spectrum, all on the same residual-graph bookkeeping.